# 🤖 02. Entrenamiento, Evaluación y Serialización de Modelos

Este notebook contiene el flujo de entrenamiento de los dos modelos de Machine Learning desarrollados para **Finance AI**:
1. **Clasificador NLP de Transacciones:** Modelo Naive Bayes para asignar categorías a partir de descripciones textuales.
2. **Predictor de Perfil de Salud Financiera:** Modelo Random Forest para clasificar el nivel de riesgo financiero de los usuarios.

## 1. Importación de Librerías

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 2. Modelo 1: Clasificador NLP de Gastos

In [2]:
# Cargar el dataset traducido en espanol
df_es = pd.read_csv('../data/raw/Personal_Finance_Dataset_ES.csv')

X_text = df_es['Transaction Description'].astype(str).tolist()
y_text = df_es['categoria'].astype(str).tolist()

# Dividir datos en entrenamiento y test para evaluar
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(X_text, y_text, test_size=0.3, random_state=42)

# Crear pipeline TF-IDF + Multinomial Naive Bayes
nlp_pipeline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB())
nlp_pipeline.fit(X_text_train, y_text_train)

### Evaluación del Clasificador NLP

In [3]:
y_text_pred = nlp_pipeline.predict(X_text_test)
print("--- Reporte de Clasificación NLP ---")
print(classification_report(y_text_test, y_text_pred))

--- Reporte de Clasificación NLP ---
                  precision    recall  f1-score   support

    Alimentación       1.00      1.00      1.00         2
    Restaurantes       1.00      1.00      1.00         2
 Salud/Bienestar       1.00      1.00      1.00         1
      Transporte       1.00      1.00      1.00         1

        accuracy                           1.00         6
       macro avg       1.00      1.00      1.00         6
    weighted avg       1.00      1.00      1.00         6



## 3. Modelo 2: Predictor del Perfil de Salud Financiera

In [4]:
# Dataset de entrenamiento (Features: [ingreso_mensual, nivel_endeudamiento_%])
X_health = np.array([
    # Perfil: Saludable
    [8000.0, 10.0],
    [6000.0, 15.0],
    [5000.0, 18.0],
    [4500.0, 12.0],
    
    # Perfil: En observación
    [4500.0, 25.0],
    [3500.0, 30.0],
    [2800.0, 35.0],
    [5000.0, 40.0],
    
    # Perfil: En riesgo
    [3000.0, 55.0],
    [2000.0, 60.0],
    [1500.0, 75.0],
    [1200.0, 80.0]
])

# Etiquetas correspondientes
y_health = np.array([
    "Saludable", "Saludable", "Saludable", "Saludable",
    "En observación", "En observación", "En observación", "En observación",
    "En riesgo", "En riesgo", "En riesgo", "En riesgo"
])

# Dividir datos para evaluación
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(X_health, y_health, test_size=0.3, random_state=42)

# Crear y entrenar el clasificador RandomForest
health_clf = RandomForestClassifier(n_estimators=100, random_state=42)
health_clf.fit(X_h_train, y_h_train)

### Evaluación del Predictor de Salud

In [5]:
y_h_pred = health_clf.predict(X_h_test)
print(f"Exactitud (Accuracy): {accuracy_score(y_h_test, y_h_pred)}")
print("\n--- Reporte de Rendimiento de Salud Financiera ---")
print(classification_report(y_h_test, y_h_pred))

Exactitud (Accuracy): 1.0

--- Reporte de Rendimiento de Salud Financiera ---
                precision    recall  f1-score   support

En observación       1.00      1.00      1.00         1
     En riesgo       1.00      1.00      1.00         2
     Saludable       1.00      1.00      1.00         1

      accuracy                           1.00         4
     macro avg       1.00      1.00      1.00         4
  weighted avg       1.00      1.00      1.00         4



## 4. Serialización de Modelos

Se exportan los modelos entrenados en binarios `.joblib` para que el backend en producción pueda cargarlos en memoria para realizar inferencias rápidas.

In [6]:
# Crear carpeta de almacenamiento si no existe
os.makedirs("../app/outbound/model_storage", exist_ok=True)

# Guardar el modelo de predicción de salud financiera
joblib.dump(health_clf, "../app/outbound/model_storage/health_model.joblib")
print("✅ Modelos guardados correctamente en '../app/outbound/model_storage/'")

✅ Modelos guardados correctamente en '../app/outbound/model_storage/'
